# Practice session 01
## Object detection audit

You receive raw predictions from an object detection model and the ground truth boxes for the same images. Your job is to audit the model with pandas and NumPy, then train a tiny model that predicts whether a detection is correct.

**Format:** 10 exercises, **30 minutes**, same style as the Grid Dynamics live coding task.

**Rules**
- Start a timer before running the first cell. Write the start and end time below.
- No AI tools. pandas / NumPy / sklearn docs are allowed, as in a real interview where you would ask the interviewer.
- Write your code in the empty cell under each exercise.
- Use exactly the variable and column names given in the task; your notebook is reviewed afterwards.
- Plots are reviewed for title, axis labels and readability.
- If you get stuck for more than 4 minutes, write a comment explaining your plan and move on. Talking through your plan is part of what the interviewer scores.

**Start time:** ____ &nbsp;&nbsp; **End time:** ____ &nbsp;&nbsp; **Exercises finished:** __ / 10

In [ ]:
# Imports you may find useful (run this cell first)
import json
import re
from io import StringIO
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
%matplotlib inline

## Data

`predictions` holds one dict per detected box. `ground_truth` holds the true boxes (at most one box per class per image). Coordinates are in pixels.

In [ ]:
predictions = [
    {'image_id': 'img_01.jpg', 'class': 'cat', 'confidence': 0.84, 'x_min': 126, 'y_min': 60, 'x_max': 360, 'y_max': 189},
    {'image_id': 'img_01.jpg', 'class': 'car', 'confidence': 0.87, 'x_min': 180, 'y_min': 163, 'x_max': 371, 'y_max': 274},
    {'image_id': 'img_01.jpg', 'class': 'person', 'confidence': 0.58, 'x_min': 147, 'y_min': 137, 'x_max': 266, 'y_max': 232},
    {'image_id': 'img_02.jpg', 'class': 'dog', 'confidence': 0.56, 'x_min': 59, 'y_min': 85, 'x_max': 242, 'y_max': 214},
    {'image_id': 'img_02.jpg', 'class': 'cat', 'confidence': 0.77, 'x_min': 194, 'y_min': 264, 'x_max': 385, 'y_max': 442},
    {'image_id': 'img_02.jpg', 'class': 'car', 'confidence': 0.35, 'x_min': 240, 'y_min': 91, 'x_max': 316, 'y_max': 204},
    {'image_id': 'img_03.jpg', 'class': 'cat', 'confidence': 0.98, 'x_min': 78, 'y_min': 79, 'x_max': 161, 'y_max': 233},
    {'image_id': 'img_03.jpg', 'class': 'dog', 'confidence': 0.83, 'x_min': 41, 'y_min': 140, 'x_max': 198, 'y_max': 267},
    {'image_id': 'img_03.jpg', 'class': 'car', 'confidence': 0.33, 'x_min': 53, 'y_min': 141, 'x_max': 105, 'y_max': 185},
    {'image_id': 'img_04.jpg', 'class': 'cat', 'confidence': 0.65, 'x_min': 245, 'y_min': 31, 'x_max': 499, 'y_max': 148},
    {'image_id': 'img_04.jpg', 'class': 'car', 'confidence': 0.66, 'x_min': 3, 'y_min': -152, 'x_max': 281, 'y_max': -48},
    {'image_id': 'img_04.jpg', 'class': 'dog', 'confidence': 0.35, 'x_min': 204, 'y_min': 111, 'x_max': 379, 'y_max': 237},
    {'image_id': 'img_05.jpg', 'class': 'car', 'confidence': 0.92, 'x_min': 189, 'y_min': 66, 'x_max': 261, 'y_max': 195},
    {'image_id': 'img_05.jpg', 'class': 'dog', 'confidence': 0.95, 'x_min': 224, 'y_min': 122, 'x_max': 344, 'y_max': 242},
    {'image_id': 'img_05.jpg', 'class': 'cat', 'confidence': 0.63, 'x_min': 34, 'y_min': 89, 'x_max': 80, 'y_max': 153},
    {'image_id': 'img_06.jpg', 'class': 'person', 'confidence': 0.81, 'x_min': 130, 'y_min': 55, 'x_max': 205, 'y_max': 279},
    {'image_id': 'img_06.jpg', 'class': 'dog', 'confidence': 0.85, 'x_min': 112, 'y_min': 33, 'x_max': 309, 'y_max': 197},
    {'image_id': 'img_06.jpg', 'class': 'cat', 'confidence': 0.68, 'x_min': 241, 'y_min': 86, 'x_max': 429, 'y_max': 175},
    {'image_id': 'img_07.jpg', 'class': 'dog', 'confidence': 0.99, 'x_min': 118, 'y_min': 181, 'x_max': 384, 'y_max': 396},
    {'image_id': 'img_07.jpg', 'class': 'cat', 'confidence': 0.87, 'x_min': 216, 'y_min': 97, 'x_max': 359, 'y_max': 175},
    {'image_id': 'img_07.jpg', 'class': 'car', 'confidence': 0.36, 'x_min': 157, 'y_min': 140, 'x_max': 341, 'y_max': 303},
    {'image_id': 'img_08.jpg', 'class': 'person', 'confidence': 0.86, 'x_min': 202, 'y_min': 39, 'x_max': 400, 'y_max': 165},
    {'image_id': 'img_08.jpg', 'class': 'car', 'confidence': 0.89, 'x_min': 82, 'y_min': 141, 'x_max': 299, 'y_max': 230},
    {'image_id': 'img_08.jpg', 'class': 'dog', 'confidence': 0.42, 'x_min': 22, 'y_min': 60, 'x_max': 121, 'y_max': 168},
]

ground_truth = [
    {'image_id': 'img_01.jpg', 'class': 'cat', 'x_min': 120, 'y_min': 44, 'x_max': 336, 'y_max': 167},
    {'image_id': 'img_01.jpg', 'class': 'car', 'x_min': 185, 'y_min': 166, 'x_max': 406, 'y_max': 277},
    {'image_id': 'img_02.jpg', 'class': 'dog', 'x_min': 35, 'y_min': 26, 'x_max': 145, 'y_max': 125},
    {'image_id': 'img_02.jpg', 'class': 'cat', 'x_min': 255, 'y_min': 174, 'x_max': 428, 'y_max': 327},
    {'image_id': 'img_03.jpg', 'class': 'cat', 'x_min': 84, 'y_min': 70, 'x_max': 162, 'y_max': 208},
    {'image_id': 'img_03.jpg', 'class': 'dog', 'x_min': 45, 'y_min': 151, 'x_max': 196, 'y_max': 275},
    {'image_id': 'img_04.jpg', 'class': 'cat', 'x_min': 272, 'y_min': 50, 'x_max': 488, 'y_max': 124},
    {'image_id': 'img_04.jpg', 'class': 'car', 'x_min': 205, 'y_min': 70, 'x_max': 414, 'y_max': 274},
    {'image_id': 'img_05.jpg', 'class': 'car', 'x_min': 189, 'y_min': 72, 'x_max': 270, 'y_max': 196},
    {'image_id': 'img_05.jpg', 'class': 'dog', 'x_min': 226, 'y_min': 111, 'x_max': 359, 'y_max': 243},
    {'image_id': 'img_06.jpg', 'class': 'person', 'x_min': 132, 'y_min': 30, 'x_max': 209, 'y_max': 227},
    {'image_id': 'img_06.jpg', 'class': 'dog', 'x_min': 137, 'y_min': 33, 'x_max': 327, 'y_max': 190},
    {'image_id': 'img_07.jpg', 'class': 'dog', 'x_min': 126, 'y_min': 187, 'x_max': 385, 'y_max': 388},
    {'image_id': 'img_07.jpg', 'class': 'cat', 'x_min': 199, 'y_min': 105, 'x_max': 354, 'y_max': 178},
    {'image_id': 'img_08.jpg', 'class': 'person', 'x_min': 230, 'y_min': 49, 'x_max': 450, 'y_max': 201},
    {'image_id': 'img_08.jpg', 'class': 'car', 'x_min': 75, 'y_min': 137, 'x_max': 302, 'y_max': 221},
]

### Exercise 1: Load the data

Convert `predictions` into a DataFrame called `df` and `ground_truth` into a DataFrame called `gt_df`. Display the first 5 rows of `df` and print its dtypes.

In [ ]:
# your code here


### Exercise 2: Box geometry

Add three columns to `df`: `width` (`x_max - x_min`), `height` (`y_max - y_min`) and `area` (`width * height`).

In [ ]:
# your code here


### Exercise 3: High-confidence detections

Create `high_conf_df` with only the rows where `confidence >= 0.80`, sorted by `confidence` descending (ties: `image_id` ascending), with a fresh `0..n-1` index.

In [ ]:
# your code here


### Exercise 4: Per-class summary

Build `class_summary`, indexed by `class`, with columns `n` (number of detections), `mean_conf` (mean confidence) and `max_area` (largest area). Sort by `n` descending, then by class name ascending.

In [ ]:
# your code here


### Exercise 5: Best detection per image

Create `top_per_image`: for each `image_id`, keep only the row with the highest confidence. One row per image, sorted by `image_id`, fresh index.

In [ ]:
# your code here


### Exercise 6: Box shape with NumPy

Without loops, add `aspect = width / height` and a column `shape` using `np.select`: `'wide'` if aspect > 1.2, `'tall'` if aspect < 0.8, otherwise `'square'`. Then create `shape_counts = df['shape'].value_counts().sort_index()`.

In [ ]:
# your code here


### Exercise 7: IoU against ground truth

Left-merge `gt_df` onto `df` on `image_id` and `class` (suffixes `('', '_gt')`), keeping the original row order. Compute the Intersection over Union with vectorised NumPy:

$$\text{IoU} = \frac{|A \cap B|}{|A| + |B| - |A \cap B|}$$

where the intersection width is $\max(0, \min(x_{max}^A, x_{max}^B) - \max(x_{min}^A, x_{min}^B))$ (same for height). Store it in `df['iou']`, with `0` for predictions that have no matching ground truth box. Finally add `is_correct = (iou >= 0.5).astype(int)`. Keep the result in `df` (drop the `_gt` columns or keep them, your choice).

In [ ]:
# your code here


### Exercise 8: Plot

Make one figure with two side-by-side subplots: (1) a boxplot of `confidence` for correct vs incorrect detections, (2) a scatter of `area` vs `iou` coloured by `class`. Add titles and axis labels. In a short comment, say whether confidence looks useful for separating correct from incorrect boxes.

In [ ]:
# your code here


### Exercise 9: Train/test split

Use features `X = df[['confidence', 'area', 'aspect']]` and target `y = df['is_correct']`. Split with `train_test_split(test_size=0.25, random_state=42, stratify=y)` into `X_train, X_test, y_train, y_test`. Print the class balance of `y_train` and `y_test`.

In [ ]:
# your code here


### Exercise 10: Logistic regression

Fit `model = LogisticRegression(max_iter=1000)` on the training set. Store the test accuracy in `acc` and print the confusion matrix. In a comment: why is accuracy alone risky on a dataset this small?

In [ ]:
# your code here


## Interviewer follow-up questions

Answer in 2 to 4 sentences each, as you would say it out loud.

1. `area` is in the hundreds of thousands while `confidence` is between 0 and 1. What does that do to logistic regression and how would you fix it?
2. Why did you use `stratify=y`, and what could happen without it on 24 rows?
3. IoU >= 0.5 is a threshold. How would you choose the confidence threshold for production if false positives are expensive?

**My answers:**

1. 
2. 
3. 

## Self-review (fill in before sending for evaluation)

- Total time:
- Exercises where I checked docs:
- Exercises I skipped or could not finish:
- What felt slow:
